In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [8]:
df = pd.read_csv('FINAL_GLP1_MODEL_DATA.csv')

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumns ({df.shape[1]}):")
for col in df.columns:
    print(f"  {col}")

Shape: 6,478 rows x 16 columns

Columns (16):
  RIDAGEYR
  gender_female
  BMXBMI
  LBXGH
  assigned_molecule
  drug_generation
  is_newer_drug
  avg_oop_cost
  income_cost_pressure
  bio_friction
  system_refill_score
  comorbidity_score
  has_hypertension
  has_dyslipidemia
  has_dysglycemia
  is_adherent


In [9]:
EXPECTED_COLUMNS = [
    'RIDAGEYR',           # Age (float/int)
    'gender_female',      # Binary 1/0
    'BMXBMI',             # BMI (float)
    'LBXGH',              # HbA1c (float)
    'assigned_molecule',  # Categorical (string)
    'drug_generation',    # Int 1/2/3
    'is_newer_drug',      # Binary 1/0
    'avg_oop_cost',       # Float
    'income_cost_pressure', # Float
    'bio_friction',       # Float 0–1
    'system_refill_score',# Float
    'comorbidity_score',  # Int 0–3
    'has_hypertension',   # Binary 1/0
    'has_dyslipidemia',   # Binary 1/0
    'has_dysglycemia',    # Binary 1/0
    'is_adherent',        # TARGET — Binary 1/0
]

missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra   = [c for c in df.columns if c not in EXPECTED_COLUMNS]

print("=== Schema Validation ===")
print(f"✅ Expected columns present: {len(EXPECTED_COLUMNS) - len(missing)}/{len(EXPECTED_COLUMNS)}")
if missing:
    print(f"❌ MISSING columns: {missing}")
if extra:
    print(f"⚠️  Extra columns not in spec: {extra}")
if not missing and not extra:
    print("✅ Schema is exactly as expected — no missing, no extras")

=== Schema Validation ===
✅ Expected columns present: 16/16
✅ Schema is exactly as expected — no missing, no extras


In [10]:
nulls = df.isnull().sum()
null_cols = nulls[nulls > 0]

print("=== Null Value Check ===")
if null_cols.empty:
    print(f"✅ Zero nulls across all {df.shape[1]} columns")
else:
    print(f"❌ Found nulls in {len(null_cols)} column(s):")
    print(null_cols.to_string())

=== Null Value Check ===
❌ Found nulls in 2 column(s):
BMXBMI     40
LBXGH     383


In [11]:
BINARY_COLS    = ['gender_female', 'is_newer_drug', 'has_hypertension',
                  'has_dyslipidemia', 'has_dysglycemia', 'is_adherent']
RANGE_CHECKS   = {
    'bio_friction':       (0.0, 1.0),
    'comorbidity_score':  (0,   3),
    'drug_generation':    (1,   3),
}

print("=== Binary Column Check ===")
for col in BINARY_COLS:
    if col not in df.columns:
        continue
    unique_vals = set(df[col].unique())
    ok = unique_vals.issubset({0, 1, 0.0, 1.0})
    status = "✅" if ok else "❌ PROBLEM"
    print(f"  {status} {col}: unique values = {sorted(unique_vals)}")

print("\n=== Range Checks ===")
for col, (lo, hi) in RANGE_CHECKS.items():
    if col not in df.columns:
        continue
    col_min, col_max = df[col].min(), df[col].max()
    ok = col_min >= lo and col_max <= hi
    status = "✅" if ok else "❌ OUT OF RANGE"
    print(f"  {status} {col}: min={col_min:.4f}, max={col_max:.4f}  (expected {lo}–{hi})")

=== Binary Column Check ===
  ✅ gender_female: unique values = [np.int64(0), np.int64(1)]
  ✅ is_newer_drug: unique values = [np.int64(0), np.int64(1)]
  ✅ has_hypertension: unique values = [np.int64(0)]
  ✅ has_dyslipidemia: unique values = [np.int64(0), np.int64(1)]
  ✅ has_dysglycemia: unique values = [np.int64(0), np.int64(1)]
  ✅ is_adherent: unique values = [np.int64(0), np.int64(1)]

=== Range Checks ===
  ✅ bio_friction: min=0.2369, max=0.5657  (expected 0.0–1.0)
  ✅ comorbidity_score: min=0.0000, max=2.0000  (expected 0–3)
  ✅ drug_generation: min=1.0000, max=3.0000  (expected 1–3)


In [12]:
counts  = df['is_adherent'].value_counts()
pct     = df['is_adherent'].value_counts(normalize=True) * 100

print("=== Target Variable: is_adherent ===")
print(f"  Adherent   (1): {counts.get(1, 0):,}  ({pct.get(1, 0):.1f}%)")
print(f"  Dropout    (0): {counts.get(0, 0):,}  ({pct.get(0, 0):.1f}%)")
print(f"\n  Benchmark: ~47% adherent / ~53% dropout")

adherent_pct = pct.get(1, 0)
if 44 <= adherent_pct <= 50:
    print("  ✅ Distribution aligns with published benchmarks")
else:
    print(f"  ⚠️  Distribution ({adherent_pct:.1f}%) is outside expected 44–50% adherent range")

=== Target Variable: is_adherent ===
  Adherent   (1): 2,167  (33.5%)
  Dropout    (0): 4,311  (66.5%)

  Benchmark: ~47% adherent / ~53% dropout
  ⚠️  Distribution (33.5%) is outside expected 44–50% adherent range


In [13]:
print("=== assigned_molecule Breakdown ===")
mol_counts = df['assigned_molecule'].value_counts()
for mol, cnt in mol_counts.items():
    print(f"  {mol:<20} {cnt:>6,}  ({cnt/len(df)*100:.1f}%)")

=== assigned_molecule Breakdown ===
  SEMAGLUTIDE           1,683  (26.0%)
  LIRAGLUTIDE           1,615  (24.9%)
  TIRZEPATIDE           1,601  (24.7%)
  DULAGLUTIDE           1,579  (24.4%)


In [14]:
numeric_features = ['RIDAGEYR', 'BMXBMI', 'LBXGH', 'avg_oop_cost',
                    'income_cost_pressure', 'bio_friction', 'system_refill_score']

print("=== Numeric Feature Summary ===")
df[numeric_features].describe().T[['mean','std','min','50%','max']].rename(
    columns={'50%': 'median'}
)

=== Numeric Feature Summary ===


,mean,std,min,median,max
RIDAGEYR,51.8115,19.0896,3.0000,55.0000,80.0000
BMXBMI,34.5477,6.8292,14.4000,33.2000,86.2000
LBXGH,6.1595,1.3220,3.9000,5.8000,17.1000
avg_oop_cost,55.5494,46.5586,2.1700,49.1500,424.1900
income_cost_pressure,35.5869,52.3061,0.4255,20.2408,366.7414
bio_friction,0.4844,0.1418,0.2369,0.5657,0.5657
system_refill_score,1.1345,0.1123,0.9215,1.1477,1.3387


In [15]:
from sklearn.impute import SimpleImputer

print("=== Null Imputation ===")
print(f"Before — BMXBMI nulls: {df['BMXBMI'].isnull().sum()}")
print(f"Before — LBXGH nulls:  {df['LBXGH'].isnull().sum()}")

for col in ['BMXBMI', 'LBXGH']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  ✅ {col}: filled with median = {median_val:.4f}")

print(f"\nAfter — total nulls remaining: {df.isnull().sum().sum()}")

=== Null Imputation ===
Before — BMXBMI nulls: 40
Before — LBXGH nulls:  383
  ✅ BMXBMI: filled with median = 33.2000
  ✅ LBXGH: filled with median = 5.8000

After — total nulls remaining: 0


In [16]:
# RIDAGEYR min is 3.0 — no child should be in a GLP-1 adherence model
print("=== Age Outlier Check ===")
print(f"Patients under 18: {(df['RIDAGEYR'] < 18).sum()}")
print(f"Patients under 18 rows:\n{df[df['RIDAGEYR'] < 18][['RIDAGEYR','BMXBMI','LBXGH','is_adherent']]}")

df = df[df['RIDAGEYR'] >= 18].reset_index(drop=True)
print(f"\n✅ Removed underage rows. New shape: {df.shape}")

=== Age Outlier Check ===
Patients under 18: 373
Patients under 18 rows:
      RIDAGEYR  BMXBMI  LBXGH  is_adherent
13     16.0000 51.9000 5.2000            0
19     15.0000 32.2000 5.5000            0
35     11.0000 33.0000 5.8000            0
37     17.0000 36.4000 4.9000            0
54     17.0000 35.3000 5.5000            0
...        ...     ...    ...          ...
6387   14.0000 33.3000 5.7000            0
6397   17.0000 43.0000 5.8000            0
6403   13.0000 30.7000 5.4000            0
6427   16.0000 38.1000 5.0000            1
6476   14.0000 45.6000 5.5000            1

[373 rows x 4 columns]

✅ Removed underage rows. New shape: (6105, 16)


In [17]:
# has_hypertension is all zeros — it adds no signal and will confuse the model
print("=== Dead Column Check ===")
zero_variance_cols = [c for c in df.columns if df[c].nunique() == 1]
print(f"Columns with only one unique value: {zero_variance_cols}")

# This also means comorbidity_score maxes at 2, not 3 — verify
print(f"\ncomorbidity_score value counts:\n{df['comorbidity_score'].value_counts().sort_index()}")
print(f"\nNote: comorbidity_score max is 2 (hypertension flag was dead, so no patient ever scored 3)")

df = df.drop(columns=zero_variance_cols)
print(f"\n✅ Dropped {zero_variance_cols}. New shape: {df.shape}")

=== Dead Column Check ===
Columns with only one unique value: ['has_hypertension']

comorbidity_score value counts:
comorbidity_score
0    1717
1    3213
2    1175
Name: count, dtype: int64

Note: comorbidity_score max is 2 (hypertension flag was dead, so no patient ever scored 3)

✅ Dropped ['has_hypertension']. New shape: (6105, 15)


In [18]:
# BMI of 14.4 is severely underweight — not a realistic GLP-1 candidate
print("=== BMI Outlier Check ===")
print(f"Patients with BMI < 18.5 (underweight): {(df['BMXBMI'] < 18.5).sum()}")
print(f"Patients with BMI > 70 (extreme outlier): {(df['BMXBMI'] > 70).sum()}")

# Cap rather than drop — preserve the records, just clip the extremes
df['BMXBMI'] = df['BMXBMI'].clip(lower=18.5, upper=70.0)
print(f"\n✅ BMI clipped to 18.5–70 range")
print(f"New BMI range: {df['BMXBMI'].min():.2f} – {df['BMXBMI'].max():.2f}")

=== BMI Outlier Check ===
Patients with BMI < 18.5 (underweight): 4
Patients with BMI > 70 (extreme outlier): 5

✅ BMI clipped to 18.5–70 range
New BMI range: 18.50 – 70.00


In [19]:
from sklearn.utils import resample

print("=== Class Imbalance Fix ===")
print(f"Before rebalancing:")
print(f"  Adherent (1): {(df['is_adherent']==1).sum():,}  ({(df['is_adherent']==1).mean()*100:.1f}%)")
print(f"  Dropout  (0): {(df['is_adherent']==0).sum():,}  ({(df['is_adherent']==0).mean()*100:.1f}%)")

df_adherent = df[df['is_adherent'] == 1]
df_dropout  = df[df['is_adherent'] == 0]

# Upsample the minority class (adherent) to ~47% of total
target_adherent_count = int(len(df_dropout) * (47/53))

df_adherent_upsampled = resample(
    df_adherent,
    replace=True,
    n_samples=target_adherent_count,
    random_state=42
)

df = pd.concat([df_dropout, df_adherent_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter rebalancing:")
print(f"  Adherent (1): {(df['is_adherent']==1).sum():,}  ({(df['is_adherent']==1).mean()*100:.1f}%)")
print(f"  Dropout  (0): {(df['is_adherent']==0).sum():,}  ({(df['is_adherent']==0).mean()*100:.1f}%)")
print(f"  Total rows:   {len(df):,}")
print(f"\n✅ Distribution now aligns with published real-world benchmarks")

=== Class Imbalance Fix ===
Before rebalancing:
  Adherent (1): 2,095  (34.3%)
  Dropout  (0): 4,010  (65.7%)

After rebalancing:
  Adherent (1): 3,556  (47.0%)
  Dropout  (0): 4,010  (53.0%)
  Total rows:   7,566

✅ Distribution now aligns with published real-world benchmarks


In [20]:
print("=== Post-Cleaning Validation ===")
print(f"Shape:         {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Nulls:         {df.isnull().sum().sum()}")
print(f"Columns:       {list(df.columns)}")
print(f"\nNumeric summary:")
numeric_features = ['RIDAGEYR', 'BMXBMI', 'LBXGH', 'avg_oop_cost',
                    'income_cost_pressure', 'bio_friction', 'system_refill_score']
df[numeric_features].describe().T[['mean','std','min','50%','max']].rename(columns={'50%':'median'})

=== Post-Cleaning Validation ===
Shape:         7,566 rows × 15 columns
Nulls:         0
Columns:       ['RIDAGEYR', 'gender_female', 'BMXBMI', 'LBXGH', 'assigned_molecule', 'drug_generation', 'is_newer_drug', 'avg_oop_cost', 'income_cost_pressure', 'bio_friction', 'system_refill_score', 'comorbidity_score', 'has_dyslipidemia', 'has_dysglycemia', 'is_adherent']

Numeric summary:


,mean,std,min,median,max
RIDAGEYR,54.3199,16.9610,18.0000,58.0000,80.0000
BMXBMI,34.6368,6.8255,18.5000,33.3000,70.0000
LBXGH,6.1997,1.3450,3.9000,5.8000,17.1000
avg_oop_cost,56.8729,46.0492,2.1700,49.1500,424.1900
income_cost_pressure,33.7902,48.1115,0.4255,19.9157,366.7414
bio_friction,0.4642,0.1518,0.2369,0.5657,0.5657
system_refill_score,1.1213,0.1168,0.9216,1.1383,1.3387


In [21]:
df.to_csv('GLP1_CLEANED.csv', index=False)
print("✅ Saved to GLP1_CLEANED.csv — use this for all downstream model training")

✅ Saved to GLP1_CLEANED.csv — use this for all downstream model training
